# Inspect Expression AnnData With BANKSY Clusters

This notebook inspects the clean expression object created from the original Xenium expression data with BANKSY cluster labels copied into `.obs`.

In [1]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

In [2]:
repo_root = Path.cwd()
if repo_root.name == "testing":
    repo_root = repo_root.parent

adata_path = repo_root / "data/xenium/processed/CK_skin_res/CK_skin_res_normalised_log1p_with_banksy_clusters_local_test.h5ad"
groupby = "labels_scaled_gaussian_pc20_nc0.20_r0.50"
annotation_key = "labels_scaled_gaussian_pc20_nc0.20_r0.50_annotation"

gene = "VWF"
cluster = "0"

In [3]:
adata = ad.read_h5ad(adata_path)
adata

AnnData object with n_obs × n_vars = 6760 × 304
    obs: 'sample_name', 'x', 'y', 'cell_area', 'nucleus_area', 'transcript_counts', 'total_counts', 'nucleus_count', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'segmentation_method', 'nCount_Xenium', 'nFeature_Xenium', 'labels_scaled_gaussian_pc20_nc0.20_r0.50', 'labels_scaled_gaussian_pc20_nc0.20_r0.50_annotation'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'log1p'
    obsm: 'spatial', 'xy'
    layers: 'counts'

## Object Checks

In [4]:
print("shape:", adata.shape)
print("raw exists:", adata.raw is not None)
print("counts layer exists:", "counts" in adata.layers)
print("groupby exists:", groupby in adata.obs.columns)
print("annotation exists:", annotation_key in adata.obs.columns)
print("gene in var_names:", gene in adata.var_names)
if adata.raw is not None:
    print("gene in raw.var_names:", gene in adata.raw.var_names)

shape: (6760, 304)
raw exists: True
counts layer exists: True
groupby exists: True
annotation exists: True
gene in var_names: True
gene in raw.var_names: True


In [5]:
adata.obs.head()

,sample_name,x,y,cell_area,nucleus_area,transcript_counts,total_counts,nucleus_count,control_probe_counts,genomic_control_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,segmentation_method,nCount_Xenium,nFeature_Xenium,labels_scaled_gaussian_pc20_nc0.20_r0.50,labels_scaled_gaussian_pc20_nc0.20_r0.50_annotation
aaabjldc-1,CK_skin_res,134.886642,638.876343,205.099695,121.831567,765,766,1,0,0,0,1,0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,765,90,0,0
aaaenhao-1,CK_skin_res,238.324539,723.201538,167.394225,108.239535,771,771,1,0,0,0,0,0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,771,110,0,0
aaamkopm-1,CK_skin_res,190.484802,734.506287,78.300940,49.220314,287,287,1,0,0,0,0,0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,287,67,0,0
aaamomnk-1,CK_skin_res,178.695374,720.238647,74.191721,52.336096,270,270,1,0,0,0,0,0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,270,65,0,0
aabbmchc-1,CK_skin_res,237.545151,570.459290,41.453439,NaN,60,60,0,0,0,0,0,0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,60,33,2,2


In [6]:
adata.var.head()

,gene_ids,feature_types,genome
ACER1,ENSG00000167769,Gene Expression,Unknown
ACTA2,ENSG00000107796,Gene Expression,Unknown
ADAM12,ENSG00000148848,Gene Expression,Unknown
AHNAK2,ENSG00000185567,Gene Expression,Unknown
AIF1,ENSG00000204472,Gene Expression,Unknown


## Cluster Labels

In [7]:
adata.obs[groupby].value_counts().sort_index()

labels_scaled_gaussian_pc20_nc0.20_r0.50
0    2172
1    1329
2    1109
3     887
4     668
5     595
Name: count, dtype: int64

In [8]:
if annotation_key in adata.obs.columns:
    display(pd.crosstab(adata.obs[groupby], adata.obs[annotation_key]))

labels_scaled_gaussian_pc20_nc0.20_r0.50_annotation,0,1,2,3,4,5
labels_scaled_gaussian_pc20_nc0.20_r0.50,,,,,,
0,2172,0,0,0,0,0
1,0,1329,0,0,0,0
2,0,0,1109,0,0,0
3,0,0,0,887,0,0
4,0,0,0,0,668,0
5,0,0,0,0,0,595


## Expression Values

In [9]:
def to_dense(x):
    if sparse.issparse(x):
        return x.toarray()
    return np.asarray(x)

print("X min/max:", float(adata.X.min()), float(adata.X.max()))
print("counts min/max:", float(adata.layers["counts"].min()), float(adata.layers["counts"].max()))
if adata.raw is not None:
    print("raw X min/max:", float(adata.raw.X.min()), float(adata.raw.X.max()))

X min/max: 0.0 5.4806389808654785
counts min/max: 0.0 304.0
raw X min/max: 0.0 5.4806389808654785


## Sanity Check Counts And Normalized Values

In [10]:
counts = adata.layers["counts"]
x = adata.X

counts_dense_sample = to_dense(counts[:100, :100])
x_dense_sample = to_dense(x[:100, :100])

print("counts layer dtype:", counts.dtype)
print("counts sample min/max:", float(counts_dense_sample.min()), float(counts_dense_sample.max()))
print("counts sample all non-negative:", bool((counts_dense_sample >= 0).all()))
print("counts sample all whole-number-like:", bool(np.allclose(counts_dense_sample, np.round(counts_dense_sample))))

print("X dtype:", x.dtype)
print("X sample min/max:", float(x_dense_sample.min()), float(x_dense_sample.max()))
print("X sample all non-negative:", bool((x_dense_sample >= 0).all()))
print("X is floating dtype:", bool(np.issubdtype(x.dtype, np.floating)))


counts layer dtype: float32
counts sample min/max: 0.0 73.0
counts sample all non-negative: True
counts sample all whole-number-like: True
X dtype: float32
X sample min/max: 0.0 4.017361640930176
X sample all non-negative: True
X is floating dtype: True


In [11]:
# Optional full-matrix checks. These may take a little longer on large objects.
counts_all_nonnegative = (counts.min() >= 0)
x_all_nonnegative = (adata.X.min() >= 0)

print("counts full matrix non-negative:", bool(counts_all_nonnegative))
print("X full matrix non-negative:", bool(x_all_nonnegative))
print("X full matrix min/max:", float(adata.X.min()), float(adata.X.max()))


counts full matrix non-negative: True
X full matrix non-negative: True
X full matrix min/max: 0.0 5.4806389808654785


In [12]:
expr = to_dense(adata[:, [gene]].X).ravel()
clusters = adata.obs[groupby].astype(str)
mask = (clusters == str(cluster)).to_numpy()
cluster_expr = expr[mask]

print("gene:", gene)
print("cluster:", cluster)
print("n_cells:", int(mask.sum()))
print("mean_expression:", float(cluster_expr.mean()))
print("percent_expressing:", float((cluster_expr > 0).mean() * 100))
cluster_expr[:20]

gene: VWF
cluster: 0
n_cells: 2172
mean_expression: 0.06793025135993958
percent_expressing: 11.694290976058932


array([0.27187145, 0.27001724, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.38495496, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.34156996, 0.4849051 ,
       0.6869296 , 0.        , 1.5507544 , 0.        , 0.        ],
      dtype=float32)

## Quick Marker Summary

In [ ]:
markers = ["VWF", "CDH5", "AIF1", "CD3D", "CD8A", "PMEL", "MLANA", "MKI67"]
present_markers = [g for g in markers if g in adata.var_names]
present_markers

In [ ]:
summary_rows = []
for cluster_id in sorted(adata.obs[groupby].astype(str).unique()):
    mask = (adata.obs[groupby].astype(str) == cluster_id).to_numpy()
    expr_mat = to_dense(adata[mask, present_markers].X)
    for i, marker in enumerate(present_markers):
        summary_rows.append({
            "cluster_id": cluster_id,
            "gene": marker,
            "mean_expression": float(expr_mat[:, i].mean()),
            "percent_expressing": float((expr_mat[:, i] > 0).mean() * 100),
            "n_cells": int(mask.sum()),
        })

pd.DataFrame(summary_rows).head(20)